# Dự báo quỹ đạo bão nhiệt đới bằng mạng nơ-ron ma trận (MNN)

Notebook này gộp toàn bộ mã của dự án vào một chỗ, chạy lần lượt từ trên xuống:

| Phần | Tương ứng file gốc |
|---|---|
| 1. Dữ liệu, chia tập | `MNN_Main.py` (`load_csv_data`, `split_by_storm`), `Kiem_tra_chia_tap.py` |
| 2. Tạo cửa sổ, vận tốc tự tính | `Data_Processing.py`, `Kinematics.py` |
| 3. Mô hình MNN | `Add_Layer.py` |
| 4. Huấn luyện | `Training.py` |
| 5. Kiểm thử | `Testing.py` |
| 6. Đánh giá chính | `MNN_Main.py` (`run_evaluation`, `compute_metrics`) |
| 7. Kiểm tra rò rỉ STORM_SPEED/STORM_DIR | (mới) |
| 8. Ablation các cấu hình đầu vào: Mean, **RMSE**, Median, P95, CI95 | `Ablation.py` |

Mô hình dùng TensorFlow 1 (chế độ `tf.compat.v1`), giống code gốc.

**Cách chạy:** đặt `ibtracs_tbd_6h.csv` cùng thư mục với notebook rồi chạy *Run All*.
Huấn luyện đầy đủ mất vài phút cho mỗi mô hình; muốn kiểm tra nhanh đường ống thì đặt `QUICK = True` ở phần 0.

## Thư viện

Notebook dùng `numpy`, `pandas`, `matplotlib`, `scikit-learn` và `tensorflow` (bản 2.x, gọi qua `tf.compat.v1`).
Chạy ô dưới để xem phiên bản đang dùng:

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import time
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf2
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior()
tf2.get_logger().setLevel('ERROR')

plt.rc('font', **{'size': 12})
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)

for name, mod in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib),
                  ('sklearn', sklearn), ('tensorflow', tf2)]:
    print(f'{name:11s} {mod.__version__}')

## 0. Cấu hình chung

Các tham số giống `MNN_Main.py`. `QUICK = True` chỉ để kiểm tra đường ống (2 epoch, 1 seed); số liệu khi đó không có ý nghĩa.

In [ ]:
SEED = 42
CSV_PATH = 'ibtracs_tbd_6h.csv'
BASE_DIR = 'KetQua_IBTrACS_MNN'      # kết quả phần 6
ABL_DIR = 'KetQua_Ablation'          # kết quả phần 8

FORECAST_LIST = [1, 2]               # số bước 6h: 1 -> +6h, 2 -> +12h
WINDOW_SIZE = 4                      # 4 bước = 24h quá khứ
FIRST_HID = [2, 25, 8]               # lớp ẩn 1: ma trận 25 x 8
SECOND_HID = [2, 50, 10]             # lớp ẩn 2: ma trận 50 x 10
TWO_STAGE = False
MAX_EPOCHS = 200
STOP_PATIENCE = 30

MAX_STEP_KM = 600.0                  # cắt đoạn nếu nhảy > 600 km / 6h
MIN_SEG_LEN = WINDOW_SIZE + max(FORECAST_LIST) + 1
NATURE_KEEP = None
MIN_MOVE_KM = 10.0

KM_PER_DEG = 111.195
COL_LAT_ABS = 5                      # cột 5 của bảng = vĩ độ tuyệt đối

QUICK = False
if QUICK:
    MAX_EPOCHS = 2

random.seed(SEED)
np.random.seed(SEED)

## 1. Dữ liệu

Tệp `ibtracs_tbd_6h.csv` có một dòng cho mỗi mốc 6h của mỗi cơn bão. Các cột dùng tới: `ID`, `ISO_TIME`, `LAT`, `LON`, `STORM_SPEED` (knot), `STORM_DIR` (độ), `NAME`, `YEAR`, `BASIN`, `NATURE`.

In [ ]:
raw = pd.read_csv(CSV_PATH, encoding='utf-8', low_memory=False, keep_default_na=False, na_values=[''])
print(raw.shape)
raw.head()

### 1.1 Cắt phân đoạn và tạo bảng cho từng phân đoạn

Mỗi cơn bão được cắt thành các phân đoạn liên tục: cắt ở chỗ hụt mốc 6h hoặc chỗ nhảy hơn `MAX_STEP_KM`. Phân đoạn ngắn hơn `MIN_SEG_LEN` điểm bị bỏ.

Mỗi phân đoạn thành một bảng 6 cột:

| Cột | Nội dung |
|---|---|
| 0 | kinh độ tương đối so với điểm đầu phân đoạn (độ) |
| 1 | vĩ độ tương đối (độ) |
| 2 | `STORM_SPEED` (knot) |
| 3, 4 | sin, cos của `STORM_DIR` |
| 5 | vĩ độ tuyệt đối |

In [ ]:
def haversine_distance(coords_true, coords_pred):
    """Khoảng cách Haversine (km); mỗi hàng là (lon, lat) theo độ."""
    R = 6371.0
    lon1, lat1 = np.radians(coords_true[:, 0]), np.radians(coords_true[:, 1])
    lon2, lat2 = np.radians(coords_pred[:, 0]), np.radians(coords_pred[:, 1])
    a = (np.sin((lat2 - lat1) / 2.0) ** 2 +
         np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def load_csv_data(csv_path, step_hours=6):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy file '{csv_path}'!")

    df = pd.read_csv(csv_path, encoding='utf-8', low_memory=False, keep_default_na=False, na_values=[''])
    df['BASIN'] = df['BASIN'].fillna('NA')
    df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'], errors='coerce')
    for c in ['STORM_SPEED', 'STORM_DIR', 'LAT', 'LON']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df[['STORM_SPEED', 'STORM_DIR']] = df.groupby('ID', sort=False)[['STORM_SPEED', 'STORM_DIR']].ffill().fillna(0.0)
    df = df.dropna(subset=['ISO_TIME', 'LAT', 'LON'])
    if NATURE_KEEP is not None:
        df = df[df['NATURE'].isin(NATURE_KEEP)]

    cyclones_data, initial_coords, meta = [], [], []
    n_jump_cuts = 0

    for storm_id, group in df.groupby('ID', sort=False):
        group = group.sort_values('ISO_TIME')
        lon = group['LON'].to_numpy(np.float64)
        lat = group['LAT'].to_numpy(np.float64)

        dt_h = group['ISO_TIME'].diff().dt.total_seconds().to_numpy() / 3600.0
        step_km = np.r_[0.0, haversine_distance(np.c_[lon[:-1], lat[:-1]], np.c_[lon[1:], lat[1:]])]
        gap = dt_h[1:] != step_hours
        jump = step_km[1:] > MAX_STEP_KM
        n_jump_cuts += int(np.sum(jump & ~gap))
        breaks = np.where(gap | jump)[0] + 1

        for seg_no, idx in enumerate(np.split(np.arange(len(group)), breaks)):
            if len(idx) < MIN_SEG_LEN:
                continue
            seg = group.iloc[idx]
            s_lon, s_lat = lon[idx], lat[idx]
            dir_rad = np.radians(seg['STORM_DIR'].to_numpy(np.float64))

            start_lon, start_lat = s_lon[0], s_lat[0]
            rel_lon = (s_lon - start_lon + 180.0) % 360.0 - 180.0
            rel_lat = s_lat - start_lat

            cols = [rel_lon, rel_lat, seg['STORM_SPEED'].to_numpy(np.float64),
                    np.sin(dir_rad), np.cos(dir_rad), s_lat]
            cyclones_data.append(np.column_stack(cols))
            initial_coords.append((start_lon, start_lat))

            name, year = str(seg['NAME'].iloc[0]).strip(), int(seg['YEAR'].iloc[0])
            meta.append({'id': storm_id, 'name': name, 'year': year, 'basin': str(seg['BASIN'].iloc[0]),
                         'nature': seg['NATURE'].astype(str).to_numpy(), 'seg': seg_no,
                         'label': f'{name}_{year}_ID{storm_id}_s{seg_no}'})
    print(f'Số chỗ cắt do nhảy > {MAX_STEP_KM:.0f} km: {n_jump_cuts}')
    return cyclones_data, initial_coords, meta

In [ ]:
all_cyc, all_init, meta = load_csv_data(CSV_PATH)
print(f'{len(all_cyc)} phân đoạn từ {len({m["id"] for m in meta})} cơn bão, '
      f'{sum(t.shape[0] for t in all_cyc)} điểm quỹ đạo')
pd.DataFrame(all_cyc[0], columns=['dlon', 'dlat', 'speed_kt', 'sin_dir', 'cos_dir', 'lat_abs']).head()

### 1.2 Chia tập train / val / test

Chia theo cơn bão (mọi phân đoạn của một cơn nằm cùng một tập), tỉ lệ 70/15/15 theo thứ tự ID như `split_by_storm` gốc.

In [ ]:
def split_by_storm(meta, f_train=0.7, f_val=0.85):
    ids = list(dict.fromkeys(m['id'] for m in meta))
    a, b = int(len(ids) * f_train), int(len(ids) * f_val)
    sets = [set(ids[:a]), set(ids[a:b]), set(ids[b:])]
    return [[i for i, m in enumerate(meta) if m['id'] in s] for s in sets]


split = split_by_storm(meta, 0.7, 0.85)
names = ['Huấn luyện', 'Kiểm định', 'Kiểm thử']

rows, id_sets = [], []
for name, idx in zip(names, split):
    ids = {meta[i]['id'] for i in idx}
    id_sets.append(ids)
    years = [meta[i]['year'] for i in idx]
    rows.append({'Tập': name, 'Số cơn bão': len(ids), 'Số phân đoạn': len(idx),
                 'Số điểm quỹ đạo': sum(all_cyc[i].shape[0] for i in idx),
                 **{f'Số cửa sổ +{6 * fs}h': sum(max(all_cyc[i].shape[0] - WINDOW_SIZE - fs, 0) for i in idx)
                    for fs in FORECAST_LIST},
                 'Năm': f'{min(years)}-{max(years)}'})
split_stats = pd.DataFrame(rows)
split_stats['Tỉ lệ cơn (%)'] = (100 * split_stats['Số cơn bão'] / split_stats['Số cơn bão'].sum()).round(2)
display(split_stats)

for a in range(3):
    for b in range(a + 1, 3):
        print(f'Trùng ID {names[a]} ∩ {names[b]}: {len(id_sets[a] & id_sets[b])} cơn')

## 2. Tạo cửa sổ đầu vào

Mỗi mẫu gồm:
- **X** (`WINDOW_SIZE` × số đặc trưng): mỗi hàng là một bước 6h. Cột 0, 1 của bảng được đổi thành độ dời km (Đông, Bắc) của bước đó; các cột khác lấy nguyên giá trị tại điểm đó.
- **Y** (1 × 2): độ dời km từ điểm neo *j* tới điểm *j + forecast_steps*.

Khác với bản gốc, `data_processing` nhận thêm `feature_cols` để chọn đặc trưng bất kỳ (ví dụ chỉ `[2, 3, 4]`), cần cho ablation.

In [ ]:
def deg_to_km(dlon, dlat, lat_ref):
    """Độ dời (độ) -> km (Đông, Bắc): dx = Δlon·cos(lat)·111, dy = Δlat·111."""
    coslat = np.cos(np.radians(lat_ref))
    return dlon * KM_PER_DEG * coslat, dlat * KM_PER_DEG


def km_to_deg(dx, dy, lat_ref):
    coslat = np.cos(np.radians(lat_ref))
    return dx / (KM_PER_DEG * coslat), dy / KM_PER_DEG


def anchor_indices(n_rows, window_size, forecast_steps=1):
    return np.arange(window_size, n_rows - forecast_steps)


def data_processing(table, window_size, forecast_steps=1, feature_cols=(0, 1, 2, 3, 4)):
    """Sinh (X, Y) theo cửa sổ trượt. Cột 0/1 trong feature_cols -> dx/dy (km); cột khác lấy nguyên."""
    table = np.asarray(table, dtype=np.float64)
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    nf = len(feature_cols)
    if len(js) == 0:
        return np.zeros((0, window_size, nf)), np.zeros((0, 1, 2))

    X = np.empty((len(js), window_size, nf))
    Y = np.empty((len(js), 1, 2))
    for n, j in enumerate(js):
        lat_ref = table[j, COL_LAT_ABS]
        steps = np.diff(table[j - window_size: j + 1, :2], axis=0)
        dx, dy = deg_to_km(steps[:, 0], steps[:, 1], lat_ref)
        for q, c in enumerate(feature_cols):
            X[n, :, q] = dx if c == 0 else dy if c == 1 else table[j - window_size + 1: j + 1, c]
        tgt = table[j + forecast_steps, :2] - table[j, :2]
        Y[n, 0] = deg_to_km(tgt[0], tgt[1], lat_ref)
    return X, Y


def persistence_km(table, window_size, forecast_steps=1):
    """Baseline persistence: lặp lại bước dời 6h gần nhất (km), tính thẳng từ bảng."""
    table = np.asarray(table, dtype=np.float64)
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    last = table[js, :2] - table[js - 1, :2]
    dx, dy = deg_to_km(last[:, 0], last[:, 1], table[js, COL_LAT_ABS])
    return forecast_steps * np.c_[dx, dy]

In [ ]:
X0, Y0 = data_processing(all_cyc[0], WINDOW_SIZE, 1)
print('X:', X0.shape, '| Y:', Y0.shape)
print('Cửa sổ đầu tiên (dx km, dy km, speed kt, sin, cos):\n', X0[0].round(2))
print('Nhãn +6h (dx, dy km):', Y0[0].round(2))

### 2.1 Speed / hướng tự tính từ tọa độ (Kinematics)

Dùng cho các cấu hình nhân quả trong ablation: giá trị tại điểm *k* chỉ dùng các điểm 0..*k*.
- `raw`: sai phân lùi.
- `kal`: bộ lọc Kalman nhân quả, trạng thái [x, y, vx, vy].
- `lag`: lấy `STORM_SPEED`/`STORM_DIR` của IBTrACS nhưng lùi 1 bước (xem phần 7 để biết vì sao).

In [ ]:
def step_km(lon, lat):
    """Độ dời km (Đông, Bắc) giữa các điểm liền kề, dùng vĩ độ trung điểm."""
    latm = np.radians((lat[1:] + lat[:-1]) / 2.0)
    dlon = (np.diff(lon) + 180.0) % 360.0 - 180.0
    return dlon * KM_PER_DEG * np.cos(latm), np.diff(lat) * KM_PER_DEG


def raw_velocity(lon, lat, dt=6.0):
    """Sai phân lùi (km/h). Hàng 0 chép từ hàng 1 (hàng 0 không bao giờ nằm trong cửa sổ)."""
    dx, dy = step_km(lon, lat)
    V = np.c_[dx, dy] / dt
    return np.vstack([V[:1], V])


def kalman_velocity(lon, lat, dt=6.0, sigma_pos=15.0, sigma_acc=0.5):
    """Kalman nhân quả; sigma_pos: sai số vị trí (km), sigma_acc: nhiễu gia tốc (km/h^2)."""
    dx, dy = step_km(lon, lat)
    pos = np.c_[np.r_[0.0, np.cumsum(dx)], np.r_[0.0, np.cumsum(dy)]]
    n = len(pos)
    A = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    Q1 = sigma_acc ** 2 * np.array([[dt ** 3 / 3, dt ** 2 / 2], [dt ** 2 / 2, dt]])
    Q = np.zeros((4, 4))
    Q[np.ix_([0, 2], [0, 2])] = Q1
    Q[np.ix_([1, 3], [1, 3])] = Q1
    H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)
    R = np.eye(2) * sigma_pos ** 2
    x = np.array([pos[0, 0], pos[0, 1], 0.0, 0.0])
    P = np.diag([sigma_pos ** 2, sigma_pos ** 2, 30.0 ** 2, 30.0 ** 2])
    V = np.zeros((n, 2))
    for k in range(1, n):
        x = A @ x
        P = A @ P @ A.T + Q
        K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
        x = x + K @ (pos[k] - H @ x)
        P = (np.eye(4) - K @ H) @ P
        V[k] = x[2:]
    V[0] = V[1]
    return V


def speed_sin_cos(V, eps=1e-6):
    """(tốc độ km/h, sin(hướng), cos(hướng)); hướng đo từ Bắc theo chiều kim đồng hồ."""
    sp = np.hypot(V[:, 0], V[:, 1])
    return sp, V[:, 0] / (sp + eps), V[:, 1] / (sp + eps)


def with_vel(t, mode, sigma_pos=15.0, sigma_acc=0.5):
    """Thay cột 2, 3, 4 bằng speed (knot), sin, cos tự tính từ tọa độ."""
    lon, lat = t[:, 0], t[:, COL_LAT_ABS]
    V = raw_velocity(lon, lat) if mode == 'raw' else kalman_velocity(lon, lat, sigma_pos=sigma_pos, sigma_acc=sigma_acc)
    sp, s, c = speed_sin_cos(V)
    t = t.copy()
    t[:, 2], t[:, 3], t[:, 4] = sp / 1.852, s, c
    return t


def lag_ib(t):
    """Lùi STORM_SPEED/DIR 1 bước -> giá trị tại j chỉ dùng điểm j-2..j."""
    t = t.copy()
    t[1:, 2:5] = t[:-1, 2:5]
    return t

## 3. Mô hình MNN

Mỗi lớp là một lớp M-neuron song tuyến tính:

$$u_{b,j,k} = \sum_{p,q} X_{b,p,q}\, W^L_{j,k,p}\, W^R_{j,k,q} + \theta_{j,k}$$

Hai lớp ẩn có LayerNorm và hàm kích hoạt algebraic $x/\sqrt{1+x^2}$; lớp ra cho ma trận 1 × 2 (dx, dy đã chuẩn hoá).

In [ ]:
def algebraic_activation(x):
    return x / tf.sqrt(1.0 + tf.square(x))


def add_layer(inputs, in_shape, out_shape, activation_function=None, use_layernorm=False,
              layer_name='m_layer', drop_rate=None):
    n1, n2 = in_shape[1], in_shape[2]
    m1, m2 = out_shape[1], out_shape[2]
    init_std = (n1 * n2) ** -0.25          # giữ phương sai đầu ra của tích song tuyến tính ~ 1

    with tf.variable_scope(layer_name):
        W = tf.Variable(tf.random.normal([m1, m2, n1], stddev=init_std), name='W_L')
        V = tf.Variable(tf.random.normal([m1, m2, n2], stddev=init_std), name='W_R')
        biases = tf.Variable(tf.zeros([1, m1, m2]), name='theta')

        u = tf.einsum('bpq,jkp,jkq->bjk', inputs, W, V) + biases

        if use_layernorm:
            mean, variance = tf.nn.moments(u, axes=[1, 2], keepdims=True)
            gamma = tf.Variable(tf.ones([1, m1, m2]), name='gamma')
            beta = tf.Variable(tf.zeros([1, m1, m2]), name='beta')
            u = gamma * ((u - mean) / tf.sqrt(variance + 1e-6)) + beta

        outputs = u if activation_function is None else activation_function(u)
        if drop_rate is not None:
            outputs = tf.nn.dropout(outputs, rate=drop_rate)
    return W, V, outputs


def build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=None):
    _, _, l1 = add_layer(xs, [None, window_size, n_feat], first_hid,
                         activation_function=algebraic_activation, use_layernorm=True,
                         layer_name='layer1', drop_rate=drop_rate)
    _, _, l2 = add_layer(l1, first_hid, second_hid,
                         activation_function=algebraic_activation, use_layernorm=True,
                         layer_name='layer2', drop_rate=drop_rate)
    _, _, prediction = add_layer(l2, second_hid, [None, 1, 2], activation_function=None,
                                 use_layernorm=False, layer_name='layer3')
    return prediction

## 4. Huấn luyện

Giống `Training.py`: Adam, lr 1e-3, batch 256, clip gradient ±1, dừng sớm khi `stop_patience` epoch liền val không giảm quá 0,2%, lưu checkpoint tại val tốt nhất.

Hai thay đổi:
- Thư mục checkpoint mặc định nằm trong thư mục làm việc (notebook không có `__file__`).
- `lr_patience` trong bản gốc khai báo nhưng không dùng (lr luôn cố định). Ở đây để mặc định `None` (giữ nguyên hành vi cũ); đặt số nguyên để bật giảm lr ×0,5 khi val chững lại.

In [ ]:
def _stack(data, window_size, forecast_steps, feature_cols):
    Xs, Ys = [], []
    for table in data:
        X, Y = data_processing(table, window_size, forecast_steps, feature_cols)
        if len(X) > 0:
            Xs.append(X)
            Ys.append(Y)
    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)


def training(data_train, data_val, window_size, first_hid, second_hid, seed=None, forecast_steps=1,
             feature_cols=(0, 1, 2, 3, 4), save_dir=None, two_stage=False, batch_size=256,
             max_epochs=200, learning_rate=0.001, lr_patience=None, stop_patience=30,
             min_rel_improve=2e-3, min_lr=1e-5, verbose=True, drop_rate=0.0, l2_lambda=0.0):
    tf.reset_default_graph()
    seed = 0 if seed is None else seed
    tf.set_random_seed(seed)
    rng = np.random.RandomState(seed)

    X_tr, Y_tr = _stack(data_train, window_size, forecast_steps, feature_cols)
    X_va, Y_va = _stack(data_val, window_size, forecast_steps, feature_cols)
    n_feat = X_tr.shape[2]

    scaler_X, scaler_Y = StandardScaler(), StandardScaler()
    scaler_X.fit(X_tr.reshape(-1, n_feat))
    scaler_Y.fit(Y_tr.reshape(-1, 2))
    sx = lambda X: scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat).astype(np.float32)
    sy = lambda Y: scaler_Y.transform(Y.reshape(-1, 2)).reshape(-1, 1, 2).astype(np.float32)
    X_tr_s, Y_tr_s, X_va_s, Y_va_s = sx(X_tr), sy(Y_tr), sx(X_va), sy(Y_va)
    if verbose:
        print(f'   Mẫu train: {len(X_tr_s)} | mẫu val: {len(X_va_s)} | số đặc trưng: {n_feat}')

    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    ys = tf.placeholder(tf.float32, [None, 1, 2])
    lr_ph = tf.placeholder(tf.float32, [])
    drop_ph = tf.placeholder_with_default(0.0, [], name='drop_rate') if drop_rate > 0 else None
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=drop_ph)
    loss = tf.reduce_mean(tf.square(ys - prediction))
    if l2_lambda > 0:
        l2 = tf.add_n([tf.nn.l2_loss(v) for v in tf.trainable_variables() if 'W_L' in v.name or 'W_R' in v.name])
        train_loss = loss + l2_lambda * l2
    else:
        train_loss = loss

    def make_train_op(var_list):
        opt = tf.train.AdamOptimizer(learning_rate=lr_ph)
        gv = opt.compute_gradients(train_loss, var_list)
        gv = [(tf.clip_by_value(g, -1.0, 1.0), v) for g, v in gv if g is not None]
        return opt.apply_gradients(gv)

    all_vars = tf.trainable_variables()
    if two_stage:
        train_ops = [make_train_op([v for v in all_vars if 'W_R' not in v.name]),
                     make_train_op([v for v in all_vars if 'W_R' in v.name])]
    else:
        train_ops = [make_train_op(all_vars)]

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, 'model')

    sess = tf.Session()
    sess.run(tf.global_variables_initializer())
    saver = tf.train.Saver(tf.trainable_variables())

    def full_loss(X, Y, chunk=8192):
        tot = 0.0
        for i in range(0, len(X), chunk):
            tot += sess.run(loss, {xs: X[i:i + chunk], ys: Y[i:i + chunk]}) * len(X[i:i + chunk])
        return tot / len(X)

    train_losses, val_losses = [], []
    best_val, best_epoch, lr = np.inf, -1, learning_rate
    ref_val, since_best, since_lr = np.inf, 0, 0
    n = len(X_tr_s)

    for epoch in range(max_epochs):
        perm = rng.permutation(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            fd = {xs: X_tr_s[b], ys: Y_tr_s[b], lr_ph: lr}
            if drop_ph is not None:
                fd[drop_ph] = drop_rate
            for op in train_ops:
                sess.run(op, fd)

        tr_l, va_l = full_loss(X_tr_s, Y_tr_s), full_loss(X_va_s, Y_va_s)
        train_losses.append(tr_l)
        val_losses.append(va_l)

        if va_l < best_val:
            best_val, best_epoch = va_l, epoch
            saver.save(sess, save_path)

        if va_l < ref_val * (1.0 - min_rel_improve):
            ref_val, since_best, since_lr = va_l, 0, 0
        else:
            since_best += 1
            since_lr += 1
            if lr_patience is not None and since_lr >= lr_patience and lr > min_lr:
                lr, since_lr = max(lr * 0.5, min_lr), 0
                if verbose:
                    print(f'   epoch {epoch}: giảm lr xuống {lr:.1e}')
            if since_best >= stop_patience:
                if verbose:
                    print(f'   Dừng sớm ở epoch {epoch}: {stop_patience} epoch liền val không giảm quá '
                          f'{100 * min_rel_improve:.1f}%')
                break

        if verbose and epoch % 10 == 0:
            print(f'   epoch {epoch:3d}  train {tr_l:.5f}  val {va_l:.5f}  lr {lr:.1e}')

    sess.close()
    if verbose:
        print(f'   Best val loss {best_val:.5f} tại epoch {best_epoch}')
    return scaler_X, scaler_Y, train_losses, val_losses, best_epoch

## 5. Kiểm thử

Khôi phục checkpoint tốt nhất, dự báo từng phân đoạn và đổi ngược về km.

In [ ]:
def testing(data_test, window_size, first_hid, second_hid, scaler_X, scaler_Y, forecast_steps=1,
            feature_cols=(0, 1, 2, 3, 4), save_dir=None):
    """Trả về (Y_true_km, Y_pred_km, valid_indices)."""
    tf.reset_default_graph()
    n_feat = scaler_X.n_features_in_
    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid)

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    sess = tf.Session()
    tf.train.Saver(tf.trainable_variables()).restore(sess, os.path.join(save_dir, 'model'))

    Y_true, Y_pred, valid = [], [], []
    for idx, storm in enumerate(data_test):
        X, Y = data_processing(storm, window_size, forecast_steps, feature_cols)
        if len(X) == 0:
            continue
        valid.append(idx)
        X_s = scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat)
        p = sess.run(prediction, feed_dict={xs: X_s})
        Y_true.append(Y.reshape(-1, 2))
        Y_pred.append(scaler_Y.inverse_transform(p.reshape(-1, 2)))
    sess.close()
    return Y_true, Y_pred, valid


def to_abs(table, initial, disp_km, forecast_steps):
    """Độ dời km từ điểm neo -> toạ độ tuyệt đối (lon, lat)."""
    js = anchor_indices(table.shape[0], WINDOW_SIZE, forecast_steps)
    anchor = table[js, :2] + np.array(initial)
    dlon, dlat = km_to_deg(disp_km[:, 0], disp_km[:, 1], table[js, COL_LAT_ABS])
    return anchor + np.c_[dlon, dlat]


def hav_errors(tables, initials, valid, true_km, pred_km, fs):
    """Sai số Haversine (km) của mọi điểm dự báo, nối theo thứ tự phân đoạn."""
    return np.concatenate([haversine_distance(to_abs(tables[i], initials[i], t, fs),
                                              to_abs(tables[i], initials[i], p, fs))
                           for i, t, p in zip(valid, true_km, pred_km)])

## 6. Đánh giá chính (như `MNN_Main.py`)

Huấn luyện mô hình chính cho từng hạn trong `FORECAST_LIST`, so với persistence. Bảng chỉ số gồm Mean, Median, **RMSE**, P95, sai số dọc/ngang quỹ đạo, sai số tốc độ, hướng và R².

`MAIN_SOURCE = 'ib'` với 5 đặc trưng là cấu hình hiện tại của `MNN_Main` (cấu hình C ở phần 8). Phần 7 cho thấy cấu hình này rò rỉ thông tin tương lai; đổi `MAIN_SOURCE = 'lag'` để dùng bản nhân quả. Đặt `RUN_MAIN = False` để bỏ qua phần này.

In [ ]:
RUN_MAIN = True
MAIN_FEATURE_COLS = [0, 1, 2, 3, 4]
MAIN_SOURCE = 'ib'          # 'ib' (như MNN_Main), 'lag', 'raw' hoặc 'kal'


def make_source(mode):
    if mode == 'ib':
        return all_cyc
    if mode == 'lag':
        return [lag_ib(t) for t in all_cyc]
    return [with_vel(t, mode) for t in all_cyc]


def compute_metrics(err_km, true_km, pred_km):
    out = {'Mean (km)': err_km.mean(), 'Median (km)': np.median(err_km),
           'RMSE (km)': np.sqrt(np.mean(err_km ** 2)), 'P95 (km)': np.percentile(err_km, 95)}
    t_len = np.linalg.norm(true_km, axis=1)
    p_len = np.linalg.norm(pred_km, axis=1)
    mv = t_len >= MIN_MOVE_KM
    t_hat = true_km[mv] / t_len[mv, None]
    e = pred_km[mv] - true_km[mv]
    ate = np.sum(e * t_hat, axis=1)
    cte = t_hat[:, 0] * e[:, 1] - t_hat[:, 1] * e[:, 0]
    cosang = np.sum(pred_km[mv] * true_km[mv], axis=1) / (p_len[mv] * t_len[mv] + 1e-9)
    out['MAE dọc quỹ đạo (km)'] = np.mean(np.abs(ate))
    out['MAE ngang quỹ đạo (km)'] = np.mean(np.abs(cte))
    out['Bias dọc quỹ đạo (km)'] = np.mean(ate)
    out['MAPE tốc độ di chuyển (%)'] = 100 * np.mean(np.abs(p_len[mv] - t_len[mv]) / t_len[mv])
    out['Sai số hướng (°)'] = np.degrees(np.mean(np.arccos(np.clip(cosang, -1, 1))))
    out['R² Δx'] = r2_score(true_km[:, 0], pred_km[:, 0])
    out['R² Δy'] = r2_score(true_km[:, 1], pred_km[:, 1])
    return out


def run_evaluation(forecast_steps, tables, feature_cols):
    hours = forecast_steps * 6
    sub_dir = os.path.join(BASE_DIR, f'DuBao_{hours}h')
    if os.path.exists(sub_dir):
        shutil.rmtree(sub_dir)
    os.makedirs(sub_dir)

    tr_idx, va_idx, te_idx = split
    pick = lambda idx: [tables[i] for i in idx]
    train_final, val_final, test_final = pick(tr_idx), pick(va_idx), pick(te_idx)
    test_init = [all_init[i] for i in te_idx]
    test_meta = [meta[i] for i in te_idx]

    print(f'\n--- CHẠY ĐÁNH GIÁ DỰ BÁO {hours}H ---')
    sX, sY, tr_losses, va_losses, best_epoch = training(
        train_final, val_final, WINDOW_SIZE, FIRST_HID, SECOND_HID, seed=SEED, forecast_steps=forecast_steps,
        feature_cols=feature_cols, two_stage=TWO_STAGE, max_epochs=MAX_EPOCHS, stop_patience=STOP_PATIENCE)
    Y_true, Y_pred, valid = testing(test_final, WINDOW_SIZE, FIRST_HID, SECOND_HID, sX, sY,
                                    forecast_steps, feature_cols)

    rows = []
    for t_km, p_km, idx in zip(Y_true, Y_pred, valid):
        table, mm, start = test_final[idx], test_meta[idx], test_init[idx]
        js = anchor_indices(table.shape[0], WINDOW_SIZE, forecast_steps)
        per_km = persistence_km(table, WINDOW_SIZE, forecast_steps)
        true_abs = to_abs(table, start, t_km, forecast_steps)
        e_mnn = haversine_distance(true_abs, to_abs(table, start, p_km, forecast_steps))
        e_per = haversine_distance(true_abs, to_abs(table, start, per_km, forecast_steps))
        for k in range(len(js)):
            rows.append({'label': mm['label'], 'name': mm['name'], 'year': mm['year'], 'basin': mm['basin'],
                         'nature': mm['nature'][js[k]], 'step': k, 'lat': table[js[k], COL_LAT_ABS],
                         'err_mnn': e_mnn[k], 'err_per': e_per[k],
                         'dx_true': t_km[k, 0], 'dy_true': t_km[k, 1], 'dx_mnn': p_km[k, 0], 'dy_mnn': p_km[k, 1],
                         'dx_per': per_km[k, 0], 'dy_per': per_km[k, 1]})

    pts = pd.DataFrame(rows)
    pts.to_csv(os.path.join(sub_dir, 'Errors_by_point.csv'), index=False)
    T = pts[['dx_true', 'dy_true']].to_numpy()
    res = pd.DataFrame([
        {'Model': 'MNN', **compute_metrics(pts['err_mnn'].to_numpy(), T, pts[['dx_mnn', 'dy_mnn']].to_numpy())},
        {'Model': 'Persistence', **compute_metrics(pts['err_per'].to_numpy(), T, pts[['dx_per', 'dy_per']].to_numpy())}])
    res['Skill Mean vs persistence (%)'] = 100 * (1 - res['Mean (km)'] / res.loc[1, 'Mean (km)'])
    res['Skill RMSE vs persistence (%)'] = 100 * (1 - res['RMSE (km)'] / res.loc[1, 'RMSE (km)'])
    res.round(3).to_csv(os.path.join(sub_dir, 'Metrics_Table.csv'), index=False, encoding='utf-8-sig')
    print(f'\nCHỈ SỐ +{hours}h trên {len(pts)} điểm dự báo:')
    display(res.set_index('Model').T.round(2))
    return {'res': res, 'pts': pts, 'train_losses': tr_losses, 'val_losses': va_losses, 'best_epoch': best_epoch}

In [ ]:
main_results = {}
if RUN_MAIN:
    os.makedirs(BASE_DIR, exist_ok=True)
    main_tables = make_source(MAIN_SOURCE)
    for fs in FORECAST_LIST:
        main_results[fs] = run_evaluation(fs, main_tables, MAIN_FEATURE_COLS)

### 6.1 Đường cong loss và phân bố sai số

In [ ]:
if main_results:
    fig, axes = plt.subplots(2, len(main_results), figsize=(6 * len(main_results), 8), squeeze=False)
    for col, (fs, r) in enumerate(main_results.items()):
        ax = axes[0, col]
        ax.plot(r['train_losses'], lw=2, label='train')
        ax.plot(r['val_losses'], lw=2, label='val')
        ax.axvline(r['best_epoch'], color='gray', ls='--', lw=1, label=f'best epoch {r["best_epoch"]}')
        ax.set_title(f'Loss (MSE chuẩn hoá) +{6 * fs}h')
        ax.set_xlabel('epoch')
        ax.legend(frameon=False)

        ax = axes[1, col]
        pts = r['pts']
        bins = np.linspace(0, np.percentile(pts['err_per'], 99), 40)
        ax.hist(pts['err_per'], bins=bins, color='lightgray', label='Persistence')
        ax.hist(pts['err_mnn'], bins=bins, histtype='step', lw=2, label='MNN')
        ax.set_title(f'Sai số Haversine +{6 * fs}h')
        ax.set_xlabel('km')
        ax.set_ylabel('số điểm')
        ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

## 7. Kiểm tra rò rỉ STORM_SPEED / STORM_DIR

So vector chuyển động suy ra từ `STORM_SPEED`/`STORM_DIR` với ba cách tính từ LAT/LON: sai phân lùi (j−1 → j), sai phân tới (j → j+1) và sai phân trung tâm ((j−1 → j+1)/2).

Nếu khớp nhất với sai phân trung tâm thì giá trị tại *j* chứa vị trí *j+1*. Khi đó bước tiếp theo dựng lại được bằng đại số: **P(j+1) − P(j) = 2·v_tâm·6h − (P(j) − P(j−1))**, tức nhãn +6h nằm sẵn trong đầu vào.

In [ ]:
d = raw.copy()
d['ISO_TIME'] = pd.to_datetime(d['ISO_TIME'], errors='coerce')
for c in ['LAT', 'LON', 'STORM_SPEED', 'STORM_DIR']:
    d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.dropna(subset=['ISO_TIME', 'LAT', 'LON']).sort_values(['ID', 'ISO_TIME']).reset_index(drop=True)
g = d.groupby('ID')


def step_xy(lat1, lon1, lat2, lon2):
    latm = np.radians((lat1 + lat2) / 2)
    return ((lon2 - lon1 + 180) % 360 - 180) * KM_PER_DEG * np.cos(latm), (lat2 - lat1) * KM_PER_DEG


ok_prev = (g['ISO_TIME'].shift(1) - d['ISO_TIME']).dt.total_seconds() == -21600
ok_next = (g['ISO_TIME'].shift(-1) - d['ISO_TIME']).dt.total_seconds() == 21600
m = (ok_prev & ok_next & d['STORM_SPEED'].notna() & d['STORM_DIR'].notna()).to_numpy()
bx, by = [v.to_numpy()[m] for v in step_xy(g['LAT'].shift(1), g['LON'].shift(1), d['LAT'], d['LON'])]
fx, fy = [v.to_numpy()[m] for v in step_xy(d['LAT'], d['LON'], g['LAT'].shift(-1), g['LON'].shift(-1))]
sp = d['STORM_SPEED'].to_numpy()[m] * 1.852 * 6          # km / 6h
dr = np.radians(d['STORM_DIR'].to_numpy()[m])
ux, uy = sp * np.sin(dr), sp * np.cos(dr)

rows = []
for name, (x, y) in {'Sai phân lùi (j-1 -> j)': (bx, by), 'Sai phân tới (j -> j+1)': (fx, fy),
                     'Sai phân trung tâm (j-1 -> j+1)/2': ((bx + fx) / 2, (by + fy) / 2)}.items():
    s = np.hypot(x, y)
    mv = s > 20
    dd = np.degrees(np.abs((np.arctan2(ux, uy) - np.arctan2(x, y) + np.pi) % (2 * np.pi) - np.pi))
    rows.append({'Cách tính': name, 'corr(speed)': np.corrcoef(sp, s)[0, 1],
                 'Sai khác vector TB (km)': np.hypot(ux - x, uy - y).mean(),
                 'Lệch hướng TB (°, bước > 20 km)': dd[mv].mean()})
display(pd.DataFrame(rows).round(3))

e_rebuild = np.hypot(2 * ux - bx - fx, 2 * uy - by - fy)
e_pers = np.hypot(bx - fx, by - fy)
print(f'Dựng lại bước j->j+1 chỉ từ đầu vào: sai số TB {e_rebuild.mean():.1f} km (trung vị {np.median(e_rebuild):.1f})')
print(f'Persistence (lặp bước lùi):         sai số TB {e_pers.mean():.1f} km')

## 8. Ablation các cấu hình đầu vào

Mỗi cấu hình được huấn luyện với nhiều seed trên **cùng** split, cùng kiến trúc và siêu tham số. Đánh giá trên tập test:

- **Mean, RMSE, Median, P95** sai số Haversine (km), lấy trung bình qua các seed; kèm độ lệch chuẩn theo seed của Mean và RMSE.
- **Skill vs persistence** tính cho cả Mean và RMSE.
- **So sánh cặp:** Δ Mean và Δ RMSE giữa hai cấu hình, khoảng tin cậy 95% bằng bootstrap theo cơn bão (lấy mẫu lại cả cơn, vì các điểm trong cùng cơn tương quan với nhau). "CI loại trừ 0" nghĩa là khác biệt có ý nghĩa thống kê.

Kết quả từng lần chạy được lưu thành `.npz` trong `KetQua_Ablation/`; chạy lại ô sẽ bỏ qua những lần đã có. Muốn chạy lại từ đầu thì xoá thư mục đó.

In [ ]:
ABL_SEEDS = [42, 43, 44]
ABL_FS = [1]                 # thêm 2 để chạy cả +12h, ví dụ [1, 2]
ABL_DROP, ABL_L2 = 0.0, 0.0  # chính quy hoá chung cho mọi cấu hình
N_BOOT = 2000

# tên -> (cột đặc trưng của bảng, nguồn speed/dir, nhân quả?, nhãn hiển thị)
ABL_CONFIGS = {
    'A':  ([0, 1],          'ib',  True,  'A: Δlon, Δlat'),
    'B':  ([2, 3, 4],       'ib',  False, 'B: speed, sin dir, cos dir (IBTrACS)'),
    'C':  ([0, 1, 2, 3, 4], 'ib',  False, 'C: cả 5 đặc trưng (speed/dir IBTrACS)'),
    "C'": ([0, 1, 2, 3, 4], 'raw', True,  "C': cả 5 đặc trưng, speed/dir tự tính lùi"),
    'D':  ([0, 1, 2, 3, 4], 'lag', True,  'D: cả 5 đặc trưng, speed/dir IBTrACS trễ 1 bước'),
    # 'E': ([0, 1, 2, 3, 4], 'kal', True, 'E: cả 5 đặc trưng, speed/dir qua Kalman nhân quả'),
}
COMPARISONS = [('B', 'A'), ('C', 'A'), ("C'", 'A'), ('C', "C'"), ('D', 'A'), ('C', 'D')]

if QUICK:
    ABL_SEEDS = [42]

os.makedirs(ABL_DIR, exist_ok=True)
SRC = {mode: make_source(mode) for mode in {v[1] for v in ABL_CONFIGS.values()}}
tr_idx, va_idx, te_idx = split

In [ ]:
def abl_file(cfg, fs, seed):
    return os.path.join(ABL_DIR, f'err_{cfg.replace(chr(39), "p")}_fs{fs}_s{seed}.npz')


def run_one(cfg, fs, seed):
    """Huấn luyện + kiểm thử một cấu hình; lưu sai số Haversine từng điểm test (và val)."""
    path = abl_file(cfg, fs, seed)
    if os.path.exists(path):
        return
    cols, src, _, _ = ABL_CONFIGS[cfg]
    T = SRC[src]
    tr, va, te = ([T[i] for i in idx] for idx in split)
    va_init, te_init = [all_init[i] for i in va_idx], [all_init[i] for i in te_idx]
    sd = os.path.join(ABL_DIR, 'ckpt', f'{cfg.replace(chr(39), "p")}_fs{fs}_s{seed}')
    t0 = time.time()
    sX, sY, trl, val, best = training(tr, va, WINDOW_SIZE, FIRST_HID, SECOND_HID, seed=seed, forecast_steps=fs,
                                      feature_cols=cols, save_dir=sd, two_stage=TWO_STAGE, max_epochs=MAX_EPOCHS,
                                      stop_patience=STOP_PATIENCE, drop_rate=ABL_DROP, l2_lambda=ABL_L2,
                                      verbose=False)
    out = {'best_epoch': best}
    for name, tabs, inits in (('val', va, va_init), ('test', te, te_init)):
        yt, yp, valid = testing(tabs, WINDOW_SIZE, FIRST_HID, SECOND_HID, sX, sY, fs, cols, save_dir=sd)
        out[f'err_{name}'] = hav_errors(tabs, inits, valid, yt, yp, fs)
    np.savez(path, **out)
    shutil.rmtree(sd, ignore_errors=True)
    e = out['err_test']
    print(f'{cfg:3s} +{6 * fs}h seed={seed}: test Mean {e.mean():6.2f} | RMSE {np.sqrt(np.mean(e ** 2)):6.2f} km '
          f'| best_epoch {best} | {time.time() - t0:.0f}s')


jobs = [(c, fs, s) for fs in ABL_FS for s in ABL_SEEDS for c in ABL_CONFIGS]
for n, (c, fs, s) in enumerate(jobs, 1):
    print(f'[{n}/{len(jobs)}] ', end='')
    if os.path.exists(abl_file(c, fs, s)):
        print(f'{c} +{6 * fs}h seed={s}: đã có, bỏ qua')
        continue
    run_one(c, fs, s)

### 8.1 Bảng so sánh các cấu hình (có RMSE)

In [ ]:
def rmse(e):
    return float(np.sqrt(np.mean(e ** 2)))


def test_groups(fs):
    """Mã cơn bão cho từng điểm test (đúng thứ tự của hav_errors) và sai số persistence."""
    te = [all_cyc[i] for i in te_idx]
    ids = np.concatenate([[meta[i]['id']] * len(anchor_indices(all_cyc[i].shape[0], WINDOW_SIZE, fs))
                          for i in te_idx])
    true_km = [data_processing(t, WINDOW_SIZE, fs, (0, 1))[1].reshape(-1, 2) for t in te]
    per_km = [persistence_km(t, WINDOW_SIZE, fs) for t in te]
    e_per = hav_errors(te, [all_init[i] for i in te_idx], range(len(te)), true_km, per_km, fs)
    return pd.factorize(ids)[0], e_per


def boot_delta(Ex, Ey, groups, n_boot=N_BOOT, seed=SEED):
    """Δ Mean, Δ RMSE (X − Y, trung bình qua seed) và CI95 bootstrap theo cơn bão.
    Ex, Ey: mảng (số seed, số điểm)."""
    G = groups.max() + 1
    cnt = np.bincount(groups, minlength=G).astype(float)
    S1 = lambda E: np.stack([np.bincount(groups, e, G) for e in E])          # (seed, G)
    S2 = lambda E: np.stack([np.bincount(groups, e ** 2, G) for e in E])
    s1x, s2x, s1y, s2y = S1(Ex), S2(Ex), S1(Ey), S2(Ey)
    w = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=n_boot).astype(float)
    nb = w @ cnt                                                              # (B,)
    d_mean = ((w @ s1x.T) - (w @ s1y.T)).mean(axis=1) / nb
    d_rmse = (np.sqrt((w @ s2x.T) / nb[:, None]) - np.sqrt((w @ s2y.T) / nb[:, None])).mean(axis=1)
    point_mean = np.mean(Ex.mean(axis=1) - Ey.mean(axis=1))
    point_rmse = np.mean(np.sqrt((Ex ** 2).mean(axis=1)) - np.sqrt((Ey ** 2).mean(axis=1)))
    return point_mean, np.percentile(d_mean, [2.5, 97.5]), point_rmse, np.percentile(d_rmse, [2.5, 97.5])


ablation_tables = {}
for fs in ABL_FS:
    groups, e_per = test_groups(fs)
    E, rows = {}, []
    for key, (cols, src, causal, label) in ABL_CONFIGS.items():
        files = [abl_file(key, fs, s) for s in ABL_SEEDS if os.path.exists(abl_file(key, fs, s))]
        if not files:
            continue
        errs = np.stack([np.load(f)['err_test'] for f in files])
        assert errs.shape[1] == len(e_per), 'số điểm test lệch nhau giữa các cấu hình'
        E[key] = errs
        means, rmses = errs.mean(axis=1), np.sqrt((errs ** 2).mean(axis=1))
        sd = (lambda v: v.std(ddof=1)) if len(files) > 1 else (lambda v: np.nan)
        rows.append({'Cấu hình': label, 'Nhân quả': causal, 'n_feat': len(cols), 'n_seed': len(files),
                     'Mean (km)': means.mean(), 'Std Mean theo seed (km)': sd(means),
                     'RMSE (km)': rmses.mean(), 'Std RMSE theo seed (km)': sd(rmses),
                     'Median (km)': np.median(errs, axis=1).mean(),
                     'P95 (km)': np.percentile(errs, 95, axis=1).mean()})
    rows.append({'Cấu hình': 'Persistence', 'Nhân quả': True, 'n_feat': np.nan, 'n_seed': np.nan,
                 'Mean (km)': e_per.mean(), 'Std Mean theo seed (km)': np.nan,
                 'RMSE (km)': rmse(e_per), 'Std RMSE theo seed (km)': np.nan,
                 'Median (km)': np.median(e_per), 'P95 (km)': np.percentile(e_per, 95)})
    tab = pd.DataFrame(rows)
    tab['Skill Mean vs persistence (%)'] = 100 * (1 - tab['Mean (km)'] / e_per.mean())
    tab['Skill RMSE vs persistence (%)'] = 100 * (1 - tab['RMSE (km)'] / rmse(e_per))

    comp = []
    for x, y in COMPARISONS:
        if x not in E or y not in E:
            continue
        dm, ci_m, dr, ci_r = boot_delta(E[x], E[y], groups)
        comp.append({'So sánh': f'{ABL_CONFIGS[x][3]}  so với  {ABL_CONFIGS[y][3]}',
                     'Δ Mean (km)': dm, 'CI95 Mean': f'[{ci_m[0]:.2f}, {ci_m[1]:.2f}]',
                     'Mean: CI loại trừ 0': bool(ci_m[0] > 0 or ci_m[1] < 0),
                     'Δ RMSE (km)': dr, 'CI95 RMSE': f'[{ci_r[0]:.2f}, {ci_r[1]:.2f}]',
                     'RMSE: CI loại trừ 0': bool(ci_r[0] > 0 or ci_r[1] < 0)})
    comp = pd.DataFrame(comp)

    tab.round(3).to_csv(os.path.join(ABL_DIR, f'Bang_cau_hinh_{6 * fs}h.csv'), index=False, encoding='utf-8-sig')
    comp.round(3).to_csv(os.path.join(ABL_DIR, f'So_sanh_cap_{6 * fs}h.csv'), index=False, encoding='utf-8-sig')
    ablation_tables[fs] = (tab, comp)

    print(f'\n========== +{6 * fs}h: {len(e_per)} điểm test, {groups.max() + 1} cơn bão ==========')
    display(tab.set_index('Cấu hình').round(2))
    print('Δ < 0: cấu hình bên trái sai số nhỏ hơn. CI95 bootstrap theo cơn bão.')
    display(comp.set_index('So sánh').round(2))
print(f'Đã lưu CSV vào {ABL_DIR}/')

### 8.2 Biểu đồ Mean và RMSE theo cấu hình

Thanh sọc là cấu hình **không nhân quả** (dùng thông tin tương lai); vạch đứt là persistence. Thanh lỗi là độ lệch chuẩn theo seed.

In [ ]:
for fs, (tab, _) in ablation_tables.items():
    models = tab[tab['Cấu hình'] != 'Persistence'].reset_index(drop=True)
    pers = tab[tab['Cấu hình'] == 'Persistence'].iloc[0]
    labels = [s.split(':')[0] for s in models['Cấu hình']]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, metric in zip(axes, ['Mean', 'RMSE']):
        vals = models[f'{metric} (km)']
        errs = models[f'Std {metric} theo seed (km)'].fillna(0)
        bars = ax.bar(labels, vals, yerr=errs, width=0.6, color='#4C78A8', edgecolor='white', capsize=4)
        for b, causal in zip(bars, models['Nhân quả']):
            if not causal:
                b.set_hatch('//')
                b.set_alpha(0.55)
        ax.axhline(pers[f'{metric} (km)'], color='#555555', ls='--', lw=1.5)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v / 2, f'{v:.1f}', ha='center', va='center',
                    color='white', fontsize=10, fontweight='bold')
        ax.set_title(f'{metric} sai số Haversine, test +{6 * fs}h')
        ax.set_ylabel('km')
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    handles = [Patch(facecolor='#4C78A8', label='Nhân quả'),
               Patch(facecolor='#4C78A8', alpha=0.55, hatch='//', edgecolor='white', label='Không nhân quả (rò rỉ)'),
               Line2D([], [], color='#555555', ls='--', lw=1.5,
                      label=f'Persistence (Mean {pers["Mean (km)"]:.1f} | RMSE {pers["RMSE (km)"]:.1f} km)')]
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()

### 8.3 Ghi chú khi đọc kết quả

- Cấu hình **B** và **C** dùng `STORM_SPEED`/`STORM_DIR` tại chính điểm neo. Phần 7 cho thấy đây là sai phân trung tâm, chứa vị trí của bước kế tiếp, nên sai số thấp của chúng không phản ánh khả năng dự báo.
- So sánh công bằng là giữa các cấu hình nhân quả (**A**, **C'**, **D**) và persistence.
- RMSE nhạy hơn Mean với các điểm sai rất xa, nên dùng cả hai: nếu một cấu hình giảm Mean mà không giảm RMSE thì nó chỉ cải thiện ở các điểm dễ.